# 04. End-to-End Pipeline

This notebook combines the earlier patterns into one runnable workflow: a clean local-document step, agents as tools, a real handoff, and a guardrail that routes ambiguous records to review.

## Why this notebook exists
The first three notebooks isolate the pieces. This one shows how they fit together in a single pipeline so you can see the full path from raw text to structured output, with a review checkpoint when the text is uncertain.

## Concepts
- Single-agent extraction
- Agents as tools
- Handoffs
- Guardrails
- Human review
- Tracing

## Dataset
Any `.txt` files stored in `data/` are loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Agents: https://openai.github.io/openai-agents-python/agents
- Tools: https://openai.github.io/openai-agents-python/tools/
- Handoffs: https://openai.github.io/openai-agents-python/handoffs/
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, handoff, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

DOCUMENTS = load_documents()
records = [{'id': item['id'], 'title': item['title'], 'text': item['text']} for item in DOCUMENTS]
records

[{'id': 1,
  'title': 'Letter 01 Madrid 1871',
  'text': 'Madrid, 4 April 1871\n\nDear brother,\n\nI have finally found a calm hour to write after the commotion of the exhibition. The galleries were crowded, and several visitors asked about the catalog, which made the day long but worthwhile.\n\nIn 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel. She also said she hoped the spring weather would make the journey easier when she next visited.\n\nWith affection,\nMaria'},
 {'id': 2,
  'title': 'Letter 02 Seville 1894',
  'text': 'Seville, 18 June 1894\n\nEsteemed colleague,\n\nThe printer in Seville announced a new edition of poems by Antonio Ruiz and asked readers to send subscriptions. The notice was meant to reach both bookshops and local reading circles, since the editor believed the edition might sell quickly.\n\nPlease advise whether the notice should also be sent to the archive copy desk. I have also enclosed a short list 

## Step 1: Single-agent extraction

Start with the simplest part of the pipeline: one agent extracting structured details from a clean record.

In [3]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Read the historical text carefully. Extract people, places, dates, and evidence. '
        'Return concise JSON-like bullet points.'
    ),
    model=DEFAULT_MODEL,
)

clean_text = records[0]['text']
with trace('End-to-end single-agent extraction'):
    extraction_result = await Runner.run(archive_agent, clean_text)
print(extraction_result.final_output)

- **People:** Maria Gomez; her brother (unnamed)
- **Places:** Madrid; Valencia
- **Date:** 4 April 1871; 1871; spring
- **Evidence:** “Madrid, 4 April 1871”; “I have finally found a calm hour to write after the commotion of the exhibition”; “Several visitors asked about the catalog”; “she hoped the spring weather would make the journey easier when she next visited”


## Step 2: Agents as tools and handoffs

Now add specialist agents. The coordinator can call them as tools, then hand off to the metadata specialist when it should take ownership of the run.

In [4]:
ocr_cleaner = Agent(
    name='OCR Cleaner',
    instructions='Fix obvious OCR mistakes while preserving original meaning. Do not invent new information.',
    model=DEFAULT_MODEL,
)

metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions='Extract people, places, dates, and key facts from cleaned historical text.',
    model=DEFAULT_MODEL,
)

summarizer = Agent(
    name='Summarization Agent',
    instructions='Write a two-sentence summary grounded in the provided cleaned text and extracted facts.',
    model=DEFAULT_MODEL,
)

cleaner_tool = ocr_cleaner.as_tool('clean_ocr', 'Clean OCR text while preserving meaning.')
extractor_tool = metadata_specialist.as_tool('extract_metadata', 'Extract structured metadata from cleaned text.')
summarizer_tool = summarizer.as_tool('summarize_history', 'Write a short summary based on cleaned text and facts.')

handoff_metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions=(
        'You receive cleaned OCR text from the coordinator. Extract people, places, dates, and key facts. '
        'Return concise evidence-based metadata.'
    ),
    model=DEFAULT_MODEL,
)

handoff_coordinator = Agent(
    name='Coordinator',
    instructions=(
        'Clean obvious OCR errors first. Then hand off to the metadata specialist. '
        'Make conservative edits and keep evidence visible.'
    ),
    tools=[cleaner_tool, extractor_tool, summarizer_tool],
    handoffs=[handoff(handoff_metadata_specialist, tool_name_override='transfer_to_metadata_specialist')],
    model=DEFAULT_MODEL,
)

handoff_coordinator

Agent(name='Coordinator', handoff_description=None, tools=[FunctionTool(name='clean_ocr', description='Clean OCR text while preserving meaning.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x71beb4139310>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='extract_metadata', description='Extract structured metadata from cleaned text.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Agen

## Step 3: Guardrail and human review

Finish by checking whether a record is clean enough to process automatically. If not, stop and send it to review.

In [5]:
@dataclass
class ReviewDecision:
    record_id: int
    uncertain: bool
    confidence: float
    notes: str

def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

def uncertainty_guardrail(ctx, agent, input_text):
    if isinstance(input_text, str) and ('may be' in input_text.lower() or 'unclear' in input_text.lower()):
        return GuardrailFunctionOutput(output_info='uncertain text', tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
    model=DEFAULT_MODEL,
)

(guardrail, review_agent)

(InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x71beab71a7a0>, name='uncertainty_guardrail', run_in_parallel=True),
 Agent(name='Review Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Assess whether the text can be safely processed. If it is uncertain, return a short note asking for human review.', prompt=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x71beab

## Step 4: Run the full pipeline

The clean record should pass through the review step and then through the multi-agent workflow. The ambiguous record should stop at the guardrail.

In [6]:
with trace('End-to-end clean pipeline'):
    clean_review = await Runner.run(review_agent, records[0]['text'])
print('Clean review result:')
print(clean_review.final_output)

with trace('End-to-end handoff pipeline'):
    pipeline_result = await Runner.run(handoff_coordinator, records[0]['text'])
print('')
print('Handoff result:')
print(pipeline_result.final_output)

Clean review result:
Safe to process.

Handoff result:
{
  "people": [
    {
      "name": "Maria Gomez",
      "role": "sender",
      "evidence": "In 1871, Maria Gomez wrote from Madrid to her brother in Valencia."
    },
    {
      "name": "brother",
      "role": "recipient",
      "evidence": "Dear brother"
    }
  ],
  "places": [
    {
      "name": "Madrid",
      "role": "origin",
      "evidence": "Madrid, 4 April 1871"
    },
    {
      "name": "Valencia",
      "role": "destination",
      "evidence": "Maria Gomez wrote from Madrid to her brother in Valencia."
    }
  ],
  "dates": [
    {
      "date": "4 April 1871",
      "evidence": "Madrid, 4 April 1871"
    },
    {
      "date": "1871",
      "evidence": "In 1871, Maria Gomez wrote from Madrid..."
    }
  ],
  "key_facts": [
    "The letter concerns an exhibition and the crowding of its galleries.",
    "Visitors asked about the catalog.",
    "Maria mentions the cost of travel as a concern.",
    "She hopes spring

In [7]:
try:
    with trace('End-to-end uncertain pipeline'):
        uncertain_review = await Runner.run(review_agent, records[-1]['text'])
    print(uncertain_review.final_output)
except Exception as exc:
    print(type(exc).__name__, exc)

Needs human review: the text is partially legible, and the transcription is uncertain.
